# RT-DETR KD — Colab A100 Training

Session management for Colab Pro+ A100 (40 GB).  
Each run saves checkpoints directly to Drive so training resumes after session drops.

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

REPO_DIR = '/content/rt_detr'
OUTPUT_BASE = '/content/drive/MyDrive/rt_detr_runs'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/umutonuryasar/RT-DETR {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

os.makedirs(OUTPUT_BASE, exist_ok=True)
%cd {REPO_DIR}

In [ ]:
# Install dependencies
!pip install -q pycocotools tensorboard

In [ ]:
# Verify GPU
!nvidia-smi

## COCO data

Run once. Skip if the Drive copy already exists.

In [ ]:
COCO_DRIVE = '/content/drive/MyDrive/coco'
COCO_LOCAL = '/content/coco'

if os.path.exists(COCO_DRIVE):
    # Symlink from Drive so scripts find data at the expected path
    if not os.path.exists(COCO_LOCAL):
        !ln -s {COCO_DRIVE} {COCO_LOCAL}
    print('COCO symlinked from Drive.')
else:
    print('COCO not found on Drive — run scripts/download_coco_subset.sh first.')

## Run configuration

Set `RUN_TAG` and `RESUME_CKPT` before each training cell.

In [ ]:
RUN_TAG        = 'run00_baseline'      # change per run
KD_TYPE        = 'none'               # none | logit | feature | combined
KD_LAMBDA      = 0.0
TEMPERATURE    = 4
KD_CFG         = ''                   # e.g. configs/kd/combined_kd.yml
TEACHER_CFG    = 'configs/rtdetr_r50vd_coco.yml'
TEACHER_WEIGHTS = ''                  # path on Drive or leave empty for baseline
EPOCHS         = 36
BATCH_SIZE     = 16                   # A100 40GB — increase if VRAM allows
IMG_SIZE       = 640

OUTPUT_DIR = f'{OUTPUT_BASE}/{RUN_TAG}'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Resume from last checkpoint if one exists
RESUME_CKPT = ''
last_ckpt = f'{OUTPUT_DIR}/checkpoint_last.pth'
if os.path.exists(last_ckpt):
    RESUME_CKPT = last_ckpt
    print(f'Resuming from {RESUME_CKPT}')
else:
    print(f'Starting fresh: {RUN_TAG}')

In [ ]:
# Build CLI flags
flags = [
    f'--student-cfg configs/rtdetr_r18vd_coco.yml',
    f'--teacher-cfg {TEACHER_CFG}',
    f'--kd-type {KD_TYPE}',
    f'--kd-lambda {KD_LAMBDA}',
    f'--temperature {TEMPERATURE}',
    f'--epochs {EPOCHS}',
    f'--batch-size {BATCH_SIZE}',
    f'--img-size {IMG_SIZE}',
    f'--output-dir {OUTPUT_DIR}',
    f'--coco-train {COCO_LOCAL}/train2017_30k',
    f'--coco-val {COCO_LOCAL}/val2017',
    f'--train-ann {COCO_LOCAL}/annotations/instances_train2017_30k.json',
    f'--val-ann {COCO_LOCAL}/annotations/instances_val2017.json',
    '--use-amp',
]

if TEACHER_WEIGHTS:
    flags.append(f'--teacher-weights {TEACHER_WEIGHTS}')
if KD_CFG:
    flags.append(f'--kd-cfg {KD_CFG}')
if RESUME_CKPT:
    flags.append(f'--resume {RESUME_CKPT}')

cmd = 'python tools/train_kd.py ' + ' '.join(flags)
print(cmd)

In [ ]:
# Launch training
!{cmd} 2>&1 | tee {OUTPUT_DIR}/train.log

## Evaluation

In [ ]:
!python tools/eval.py \
    --cfg configs/rtdetr_r18vd_coco.yml \
    --weights {OUTPUT_DIR}/checkpoint_best.pth \
    --coco-val {COCO_LOCAL}/val2017 \
    --val-ann {COCO_LOCAL}/annotations/instances_val2017.json \
    --img-size {IMG_SIZE} \
    2>&1 | tee {OUTPUT_DIR}/eval.log

In [ ]:
!python tools/benchmark_fps.py \
    --cfg configs/rtdetr_r18vd_coco.yml \
    --weights {OUTPUT_DIR}/checkpoint_best.pth \
    --input-size {IMG_SIZE} \
    --warmup 50 \
    --iters 500 \
    --device cuda \
    2>&1 | tee {OUTPUT_DIR}/fps.log